## Project Objective
In this notebook, we perform feature engineering on the cleaned RetailRocket e-commerce dataset.

The primary objective of this notebook is to:

- Transform raw interactions into machine learning-ready features
- Build user-level behavioral features
- Build item-level popularity and engagement features
- Prepare structured input for recommendation algorithms
- Align dataset with modular src/ pipeline scripts

This notebook strictly focuses on feature engineering only and follows a modular ML architecture where core logic is also implemented in reusable .py scripts inside the src/ folder.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import sys
import os 
project_root = os.path.abspath("..")
sys.path.append(project_root)
print(project_root)

In [ ]:
# Load cleaned datasets
from src.database import get_engine
import pandas as pd
import numpy as np

engine = get_engine()

events_df = pd.read_sql(
    "SELECT * FROM events_cleaned",
    engine
)

category_tree_df = pd.read_sql(
    "SELECT * FROM category_tree_cleaned",
    engine
)

print(events_df.shape)
print(category_tree_df.shape)

In [4]:
#timestamp
events_df["timestamp"] = pd.to_datetime(
    events_df["timestamp"]
)

In [5]:
#Convert categorical interaction types into numerical signals
interaction_weights = {
    "view": 1,
    "addtocart": 3,
    "transaction": 5
}

events_df["interaction_strength"] = (
    events_df["event"]
    .map(interaction_weights)
)

In [6]:
#reference date
reference_date = (
    events_df["timestamp"]
    .max()
)

### Customer Features

In [7]:
#Total Interactions per User
total_interactions = (
    events_df
    .groupby("visitorid")
    .size()
    .reset_index(
        name="total_interactions"
    )
)

In [8]:
total_views = (
    events_df[events_df["event"] == "view"]
    .groupby("visitorid")
    .size()
    .reset_index(
        name="total_views"
    )
)

In [9]:
total_cart = (
    events_df[
        events_df["event"] == "addtocart"
    ]
    .groupby("visitorid")
    .size()
    .reset_index(
        name="total_cart"
    )
)

In [10]:
total_transactions = (
    events_df[
        events_df["event"] == "transaction"
    ]
    .groupby("visitorid")
    .size()
    .reset_index(
        name="total_transactions"
    )
)

In [11]:
#total unique items
unique_items = (
    events_df
    .groupby("visitorid")["itemid"]
    .nunique()
    .reset_index(
        name="unique_items"
    )
)

In [12]:
#average interaction strength
avg_strength = (
    events_df
    .groupby("visitorid")
    ["interaction_strength"]
    .mean()
    .reset_index(
        name="avg_interaction_strength"
    )
)

In [13]:
#recency
last_activity = (
    events_df
    .groupby("visitorid")
    ["timestamp"]
    .max()
    .reset_index()
)

last_activity["recency_days"] = (
    reference_date
    - last_activity["timestamp"]
).dt.days

### merge features

In [14]:
#one by one merge 
customer_features = (
    total_interactions
    .merge(total_views,
           on="visitorid",
           how="left")
    .merge(total_cart,
           on="visitorid",
           how="left")
    .merge(total_transactions,
           on="visitorid",
           how="left")
    .merge(unique_items,
           on="visitorid",
           how="left")
    .merge(avg_strength,
           on="visitorid",
           how="left")
    .merge(
        last_activity[
            ["visitorid",
             "recency_days"]
        ],
        on="visitorid",
        how="left"
    )
)

In [15]:
customer_features.fillna(0,
                         inplace=True)

In [ ]:
customer_features.info()

In [ ]:
customer_features.head()

In [ ]:
customer_features.describe()

# Create User-Item Interaction Features

## Objective

Build a user-item interaction dataset that will serve as the input to the Recommendation Engine.

Unlike customer-level features, this dataset preserves interactions between individual users and products, making it suitable for collaborative filtering algorithms.

In [19]:
user_item_features = (
    events_df
    .groupby(["visitorid", "itemid"])
    .agg(
        interaction_strength=(
            "interaction_strength",
            "sum"
        )
    )
    .reset_index()
)

In [ ]:
print(user_item_features.shape)

user_item_features.head()

In [ ]:
#saving data into csv 
customer_features.to_csv(
    "../data/features/customer_features.csv",
    index=False
)
user_item_features.to_csv(
    "../data/features/user_item_features.csv",
    index=False
)
print("Feature datasets saved successfully.")

# Conclusion

In this notebook, we successfully transformed raw customer interaction data into meaningful behavioral features for machine learning applications.

### Feature Engineering Tasks Completed

- Created interaction strength scores based on user actions
- Calculated total customer interactions
- Generated total view counts
- Generated add-to-cart counts
- Generated transaction counts
- Calculated unique items interacted with
- Computed average interaction strength
- Created recency-based features using customer activity timestamps
- Merged all engineered features into a unified customer-level dataset
- Performed missing value handling and validation checks

### Output Generated

The final dataset:

`customer_features.csv`

contains customer-level behavioral features that summarize user engagement and purchasing patterns.

### Business Value

These engineered features provide a compact representation of customer behavior and will serve as the foundation for:

- Customer Segmentation
- Personalized Recommendations
- Customer Profiling
- Marketing Strategy Development

### Next Step

The engineered customer features will be used in the next notebook:

